## Jupyter kernel from the pyQuil container (OnDemand)

This notebook registers a Jupyter kernel that runs Python **inside** the
`pyquil.sif` Apptainer container, so the other notebooks in this folder get
pyQuil 4, forest-benchmarking and Rigetti's `quilc` compiler and `qvm`
simulator (Forest SDK 2.23.0) without a local virtual environment:

| kernel | Apptainer flags | use |
|---|---|---|
| **Python (pyQuil container)** | `--cleanenv --bind /ocean` | any CPU node |

Run it **once**, with the default **Python 3 (ipykernel)** kernel selected in
the top-right corner. The kernel is saved in
`~/.local/share/jupyter/kernels/` and stays available in later sessions.

The container was built from `pyquil.def` in this repository
(`apptainer build --fakeroot pyquil.sif pyquil.def`).

##### 1. Check that the container starts and has `ipykernel`, `pyquil`, `quilc` and `qvm`

In [ ]:
# `!` runs a shell command.  `apptainer exec` starts the container and runs
# a command inside it; ipykernel is what lets Jupyter talk to that Python.
SIF = "/ocean/projects/pscstaff/nnepal/Rigetti/pyquil.sif"   # change if the image is elsewhere
!apptainer exec --cleanenv {SIF} python -c "import ipykernel, pyquil, forest.benchmarking; print('ipykernel', ipykernel.__version__, '| pyquil', pyquil.__version__)"
!apptainer exec --cleanenv {SIF} quilc --version
!apptainer exec --cleanenv {SIF} qvm --version

##### 2. Register the kernel

In [ ]:
import json, os, shutil   # standard library

sif    = SIF
runner = shutil.which("apptainer")           # full path of the apptainer command
assert runner and os.path.exists(sif), "apptainer or the container image is missing"

name, display = "pyquil-container", "Python (pyQuil container)"
# Jupyter finds user kernels in ~/.local/share/jupyter/kernels/<name>/kernel.json
kdir = os.path.expanduser(f"~/.local/share/jupyter/kernels/{name}")
os.makedirs(kdir, exist_ok=True)
# The command Jupyter runs to start the kernel:
#   --cleanenv            do not pass the host environment (OnDemand's Python
#                         paths, LD_LIBRARY_PATH) into the container
#   --bind /ocean         make project folders readable/writable
#   PYTHONNOUSERSITE=1    ignore packages in ~/.local, use the container's only
#   {connection_file}     Jupyter fills this in with the kernel's port file
argv = [runner, "exec", "--cleanenv",
        "--bind", "/ocean",
        "--env", "PYTHONNOUSERSITE=1",
        sif, "python", "-m", "ipykernel_launcher", "-f", "{connection_file}"]
with open(f"{kdir}/kernel.json", "w") as f:
    json.dump({"argv": argv, "display_name": display, "language": "python"}, f, indent=2)
print("created", name, "->", kdir)

From now on, select **Python (pyQuil container)** in the top-right corner
(refresh the page if it is not listed yet).

Inside this kernel `quilc` and `qvm` are already on `PATH`, and the container
provides `libzmq.so` and BLAS/LAPACK, so the `pyzmq<26` / `LD_LIBRARY_PATH`
steps and `module load openblas` from the README are **not** needed.

> **Caution:** pyQuil reaches the servers at `127.0.0.1`, so they must run on
> the same node as the kernel. Starting them from the kernel (cell below)
> guarantees that. `--cleanenv` also removes the Slurm variables.

##### 3. Check the new kernel

Switch to **Python (pyQuil container)**, then run the cells below. The first
starts `qvm` (port 5000) and `quilc` (port 5555) inside the container in the
background; the second runs a Bell state and should print 10 rows that are each
`[0 0]` or `[1 1]`.

In [ ]:
import subprocess, socket, time, sys

def port_open(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

for cmd, port in (("qvm -S", 5000), ("quilc -P -S", 5555)):
    name = cmd.split()[0]
    if port_open(port):
        print(f"{name}: port {port} already in use, assuming it is running")
        continue
    subprocess.Popen(cmd.split(), stdout=open(f"{name}.log", "w"),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        if port_open(port):
            break
        time.sleep(1)
    print(f"{name}: {'running' if port_open(port) else 'NOT running, see ' + name + '.log'} (port {port})")
print(sys.executable)

In [ ]:
from pyquil import Program, get_qc
from pyquil.gates import H, CNOT

qc = get_qc("2q-qvm")
p = Program(H(0), CNOT(0, 1))
p.measure_all()
p.wrap_in_numshots_loop(10)

result = qc.run(qc.compile(p))
print(result.get_register_map()["ro"])

Stop the servers when you are done (or just shut the kernel down / end the
OnDemand session):

In [ ]:
!pkill -u $(id -u) -f "qvm -S"; pkill -u $(id -u) -f "quilc -P -S"; echo stopped